In [2]:
%reload_ext autoreload
%autoreload 2

import warnings, os, sys
warnings.filterwarnings('ignore')
from IPython.display import clear_output
from pathlib import Path

import numpy as np
import pandas as pd
import pickle
from pandarallel import pandarallel
pandarallel.initialize()
from tqdm.auto import tqdm
import json
import networkx as nx
from networkx.readwrite import json_graph

from rdkit import Chem
from rdkit.Chem import PandasTools
# PandasTools.RenderImagesInAllDataFrames(images=True)
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')

from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.ipython_useSVG = True
IPythonConsole.molSize = 700, 500

sys.path.append(os.path.abspath(".."))
from new_geass import Geass
import utils
clear_output()

In [2]:
df_NPA = pd.read_csv('../data/NPAtlas_download_2024_09.tsv', sep='\t')
PandasTools.AddMoleculeColumnToFrame(df_NPA, 'compound_smiles')
df_NPA = df_NPA.dropna(subset='ROMol')

allowed_elements = {"H", "C", "N", "O", "F", "P", "S", "Cl", "Br", "I"}
df_NPA = df_NPA[df_NPA["ROMol"].apply(lambda mol: all(
    atom.GetSymbol() in allowed_elements for atom in mol.GetAtoms()))].reset_index(drop=True)

df_NPA['ROMol'] = df_NPA['ROMol'].parallel_map(utils.standardize_mol)
df_NPA["ISOSMILES"] = df_NPA["ROMol"].parallel_map(lambda x: Chem.MolToSmiles(x))
df_NPA = df_NPA.drop_duplicates(subset='ISOSMILES').reset_index(drop=True)
clear_output()

In [4]:
_GEASS = None

def process_row(row):
    global _GEASS

    if _GEASS is None:
        _GEASS = Geass(method='map4', sim_threshold=0.01)

    try:
        mol = Chem.MolFromSmiles(row.compound_smiles)
        frags = _GEASS.get_fragments(mol)

        frag_str = ",".join(Chem.MolToSmiles(x) for x in frags)
        mu_str = ",".join(x.GetProp("MUID") for x in frags)
        sim_str = ",".join(x.GetProp("Similarity") for x in frags)
        ann_str = ",".join(x.GetProp("Annotation") for x in frags)

        # g = _GEASS.create_connection_graph(img=False)
        # graph_str = json.dumps(json_graph.node_link_data(g))

    except Exception:
        frag_str, mu_str, sim_str, ann_str = "ERROR", "", "", ""
        # graph_str = ""

    full_str = f'{row.npaid},{row.compound_smiles},"[{frag_str}]","[{mu_str}]","[{sim_str}]","[{ann_str}]"\n'
    # graph_line = f"{row.npaid}\t{graph_str}\n"

    return full_str#, graph_line

In [6]:
frag_path = Path("NPA_fragments_map4.csv")
# graph_path = Path("NPA_graphs_map4.jsonl")
frag_path.unlink(missing_ok=True)
# graph_path.unlink(missing_ok=True)

with open(frag_path, "a") as f_frag:#, open(graph_path, "a") as f_graph:
    if frag_path.stat().st_size == 0:
        f_frag.write("NPA_ID,ISOSMILES,Frag_ISOSMILES,Frag_IDs,Frag_Sim,Frag_Ann\n")

    chunks = np.array_split(df_NPA, 50)
    for chunk in tqdm(chunks, total=len(chunks)):
        results = chunk.parallel_apply(process_row, axis=1)

        frag_buffer = []
        # graph_buffer = []

        for full_str in results:
            frag_buffer.append(full_str)
            # graph_buffer.append(graph_line)

        if frag_buffer:
            f_frag.writelines(frag_buffer)
            # f_graph.writelines(graph_buffer)

  0%|          | 0/50 [00:00<?, ?it/s]

In [7]:
df_frags = pd.read_csv('NPA_fragments_map4.csv')
frag_cols = ['Frag_IDs', 'Frag_ISOSMILES', 'Frag_Sim', 'Frag_Ann']
df_frags[frag_cols] = df_frags[frag_cols].apply(
    lambda s: s.str[1:-1].str.split(","))
df_frags['Frag_Sim'] = df_frags['Frag_Sim'].apply(lambda x: [float(y) for y in x])
df_frags.to_pickle("NPA_fragments_map4.pkl")

In [ ]:
# Reconstruct graph objects and annotate with unknown IDs
from networkx.readwrite import json_graph
graphs = {}
with open("NPA_graphs_map4.jsonl", "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split("\t", 1)
        if len(parts) != 2:
            continue
        npaid, g_json = parts
        data = json.loads(g_json)
        graphs[npaid] = json_graph.node_link_graph(data)

with open("NPA_graphs_map4.pkl", "wb") as f:
    pickle.dump(graphs, f, protocol=pickle.HIGHEST_PROTOCOL)

## Calculate MAP4 fingerprints 

## Build a local library of annotated fragments 

In [11]:
pd.read_pickle('../data/MUDB.pkl')

,ID,ISOSMILES,Formula,MW,Num_Atoms,InChIKey,Ontology
0,C00001,O,H2O,18.015,1,XLYOFNOQVPJJNP-UHFFFAOYSA-N,Therapeutic category of drugs in Japan [BR:br0...
1,C00002,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)OP(=O)(O...,C10H16N5O13P3,507.181,31,ZKHQWZAMYRWXGA-KQYNXXCUSA-N,Compounds with biological roles [BR:br08001]\n...
2,C00003,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,C21H28N7O14P2+,664.433,44,BAWFJGJZGIEFAR-NNYOXOHSSA-O,Compounds with biological roles [BR:br08001]\n...
3,C00004,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,C21H29N7O14P2,665.441,44,BOPGDPNILDQYTO-NNYOXOHSSA-N,None
4,C00005,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,C21H30N7O17P3,745.421,48,ACFIXJIJDZMPPO-NNYOXOHSSA-N,None
...,...,...,...,...,...,...,...
61679,LMPK15050002,CCCCCCCCCCC[C@H](CC1=C(O)C(=O)C=C(OC)C1=O)OC(C)=O,C22H34O6,394.508,28,ZYWAVLTZNSYXDF-QGZVFWFLSA-N,Phenolic lipids [PK15]; Alkyl hydroquinones an...
61680,LMPK15050003,CCCCCCCCCCCCC[C@H](CC1=C(O)C(=O)C=C(OC)C1=O)OC...,C24H38O6,422.562,30,UXLMJHNFDRMGPW-LJQANCHMSA-N,Phenolic lipids [PK15]; Alkyl hydroquinones an...
61681,LMPK15050004,CCCCCCCCCCCCC[C@H](CC1=CC(=O)C=C(OC)C1=O)OC(C)=O,C24H38O5,406.563,29,CVZNKLNAHBTINT-JOCHJYFZSA-N,Phenolic lipids [PK15]; Alkyl hydroquinones an...
61682,LMPK15050005,CCCCCCCCCCC[C@H](CC1=CC(=O)C=C(OC)C1=O)OC(C)=O,C22H34O5,378.509,27,JIUGZSYPFREDLG-HXUWFJFHSA-N,Phenolic lipids [PK15]; Alkyl hydroquinones an...


In [16]:
df_fragments = pd.read_pickle('../NPA_fragmentation/NPA_fragments_map4.pkl')
df_lib = df_fragments[df_fragments['Frag_IDs'].apply(lambda x: len(x) > 1)]
cols = ["Frag_ISOSMILES", "Frag_IDs", "Frag_Sim", "Frag_Ann"]
df_lib = df_lib.explode(cols).drop_duplicates(subset="Frag_ISOSMILES")
df_lib = df_lib[df_lib["Frag_Ann"] == "Similarity"]
df_lib = df_lib[cols].reset_index(drop=True)

df_MUDB = pd.read_pickle('../data/MUDB.pkl')
df_lib = df_lib.merge(df_MUDB[['ID', 'ISOSMILES']],
    left_on='Frag_IDs', right_on='ID', how='left')
df_lib = df_lib.rename(columns={'Frag_ISOSMILES': 'ISOSMILES',
    'Frag_IDs': 'MUID_Similar', 'Frag_Sim': 'Similarity', 'Frag_Ann': 'Annotation',
    'ISOSMILES': 'ISOSMILES_Similar'})
df_lib = df_lib[['ISOSMILES', 'MUID_Similar', 'ISOSMILES_Similar', 'Similarity', 'Annotation']]

df_lib.to_csv("NPA_library.csv", index=None)